# Imports

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F , torchvision
from torch.utils.data import DataLoader , Subset
from torchvision import transforms,datasets,models
import numpy as np
import matplotlib.pyplot as plt

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
#@title Variable
BATCH_SIZE = 12
IMG_SIZE = 28
PATCH_SIZE = 7
ATTENTION_HEADS = 4
EMBED_DIM = 32
CLASSES = 10
EPOCHS_STUDENT = 5
LR_STUDENT = 3e-4
CHANNELS = 3
TRANSFORMER_LAYERS = 4
TEMPERATURE = 2
ALPHA = 0.5

In [9]:
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda t : t.repeat(3,1,1))
])

In [10]:
train_ds = datasets.MNIST(root = './data' , train = True , download = True , transform = data_transform)
val_ds = datasets.MNIST(root = './data' , train = False , download = True , transform = data_transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 37.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.06MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.16MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.01MB/s]


Create Train And Val batches

In [11]:
train_dl = DataLoader(train_ds,batch_size = BATCH_SIZE , shuffle = True)
val_dl = DataLoader(val_ds,batch_size = BATCH_SIZE )

In [12]:
teacher = models.resnet50(weights = models.ResNet50_Weights.IMAGENET1K_V2)
teacher.fc = nn.Linear(teacher.fc.in_features,CLASSES)
teacher.to(device)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s] 


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [13]:
#@Student ViT
#Patch_Embed
class PatchEmbed(nn.Module):
    def __init__(self,patch_size = PATCH_SIZE,img_size = IMG_SIZE):
        super().__init__()
        self.proj = nn.Conv2d(CHANNELS,EMBED_DIM,kernel_size = patch_size , stride = patch_size)
        self.n = (img_size // patch_size) ** 2
        
    def forward(self,img):
        img = self.proj(img)
        img = img.flatten(start_dim = 2)
        img = img.transpose(1,2)
        return img

In [14]:
class TransformerBlocks(nn.Module):
    def __init__(self,embed_dim = EMBED_DIM,attn_head = ATTENTION_HEADS):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim = embed_dim,num_heads = attn_head , batch_first = True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim,embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4 , embed_dim)
        )

    def forward(self,x):
        res1 = x
        x = self.norm1(x)
        x = self.mha(x,x,x)[0]
        x += res1
        res2 = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = x + res2
        return x

In [15]:
#VitClass

class Vit(nn.Module):
    def __init__(self,embed_dim = EMBED_DIM,img_size = IMG_SIZE , patch_size = PATCH_SIZE , attn_head = ATTENTION_HEADS,classes = CLASSES):
        super().__init__()
        self.patch = PatchEmbed()
        self.cls = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.distil = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos = nn.Parameter(torch.zeros(1, ((img_size // patch_size) ** 2) + 2 ,embed_dim))
        self.transformer_blocks = nn.Sequential(*[TransformerBlocks(embed_dim,attn_head) for _ in range(TRANSFORMER_LAYERS)])
        self.layernorm = nn.LayerNorm(embed_dim)
        self.cls_head = nn.Linear(embed_dim,classes)
        self.distil_head = nn.Linear(embed_dim,classes)
        
    def forward(self,x):
        x = self.patch(x)
        B = x.shape[0]
        cls = self.cls.expand(B,-1,-1)
        distil = self.distil.expand(B,-1,-1)
        x = torch.concat((cls,x,distil),dim = 1)
        x += self.pos
        x = self.transformer_blocks(x)
        x = self.layernorm(x)
        cls_token = x[:,0]
        distil_token = x[:,-1]
        cls_logits = self.cls_head(cls_token)
        distil_logits = self.distil_head(distil_token)
        
        return cls_logits,distil_logits

In [16]:
student = Vit().to(device)
opt_s = torch.optim.AdamW(student.parameters() , lr = LR_STUDENT)

In [17]:
# ORIGINAL CODE (kept for learning/reference):
# def kd_loss(c_logits,d_logits,t_logits,y,alpha = ALPHA,temperature = TEMPERATURE):
#     kd = F.kl_div(F.softmax(distil_logits/temperature,dim = 1),
#                  F.softmax(t_logits/temperature,dim = 1,reduction = 'batchmean') * (temperature**2))
#     ce = F.cross_entropy(c_logits,y)
#     return alpha * kd + (1 - alpha) * ce

# CHANGED: The original KD loss has a few issues:
# 1. distil_logits is not an argument; d_logits is the argument.
# 2. F.kl_div expects the student's input as log-probabilities.
# 3. reduction='batchmean' belongs to F.kl_div, not F.softmax.
# 4. For DeiT, the distillation token learns from the teacher, while the class
#    token learns from the hard labels.

def kd_loss(c_logits, d_logits, t_logits, y,alpha = ALPHA, temperature = TEMPERATURE):

    # CHANGED: class token -> normal supervised cross entropy
    cls_loss = F.cross_entropy(c_logits, y)

    # CHANGED: distillation token -> teacher soft targets
    student_log_probs = F.log_softmax(d_logits / temperature, dim=1)
    teacher_probs = F.softmax(t_logits / temperature, dim=1)

    kd = F.kl_div(student_log_probs,teacher_probs,reduction='batchmean') * (temperature ** 2)

    # CHANGED: combine the hard-label loss and teacher distillation loss
    return (1 - alpha) * cls_loss + alpha * kd


## Train the Teacher (ResNet50) ADDED: Your teacher must be trained on MNIST before using it to teach the student. The ImageNet pretrained backbone gives us a useful initialization, but the new 10-class head is randomly initialized.

In [18]:
# ADDED: Teacher training configuration
EPOCHS_TEACHER = 3
LR_TEACHER = 1e-4

# ADDED: CrossEntropy is enough for the teacher.
teacher_criterion = nn.CrossEntropyLoss()

# ADDED: train only parameters that require gradients
opt_t = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, teacher.parameters()),
    lr=LR_TEACHER
)


In [19]:
# ADDED: Generic accuracy function
def accuracy_from_logits(logits, labels):
    predictions = logits.argmax(dim=1)
    return (predictions == labels).sum().item()


# ADDED: Train the teacher
for epoch in range(EPOCHS_TEACHER):
    teacher.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_dl:
        images = images.to(device)
        labels = labels.to(device)

        opt_t.zero_grad()

        logits = teacher(images)
        loss = teacher_criterion(logits, labels)

        loss.backward()
        opt_t.step()

        running_loss += loss.item() * images.size(0)
        correct += accuracy_from_logits(logits, labels)
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100 * correct / total

    # ADDED: teacher validation accuracy
    teacher.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_dl:
            images = images.to(device)
            labels = labels.to(device)

            logits = teacher(images)
            loss = teacher_criterion(logits, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += accuracy_from_logits(logits, labels)
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    print(
        f"Teacher Epoch [{epoch+1}/{EPOCHS_TEACHER}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
    )


Teacher Epoch [1/3] Train Loss: 0.4089 | Train Acc: 87.68% | Val Loss: 0.0576 | Val Acc: 98.25%
Teacher Epoch [2/3] Train Loss: 0.0876 | Train Acc: 97.62% | Val Loss: 0.0318 | Val Acc: 99.09%
Teacher Epoch [3/3] Train Loss: 0.0576 | Train Acc: 98.52% | Val Loss: 0.0384 | Val Acc: 98.86%


# Train the DeiT Student
# ADDED: The teacher is frozen here. Only the student is updated.The class token is trained using the real labels, while the distillation token learns from the teacher.

In [20]:
# ADDED: Freeze teacher during knowledge distillation
teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False

# CHANGED: recreate the optimizer after the teacher is frozen.
opt_s = torch.optim.AdamW(student.parameters(), lr=LR_STUDENT)


In [21]:
# ADDED: Student training + validation loop

for epoch in range(EPOCHS_STUDENT):
    student.train()

    running_loss = 0.0
    cls_correct = 0
    distil_correct = 0
    total = 0

    for images, labels in train_dl:
        images = images.to(device)
        labels = labels.to(device)

        opt_s.zero_grad()

        # Teacher does not receive gradients.
        with torch.no_grad():
            teacher_logits = teacher(images)

        # Student produces two outputs:
        # cls_logits    -> class token
        # distil_logits -> distillation token
        cls_logits, distil_logits = student(images)

        loss = kd_loss(
            cls_logits,
            distil_logits,
            teacher_logits,
            labels
        )

        loss.backward()
        opt_s.step()

        running_loss += loss.item() * images.size(0)

        cls_correct += accuracy_from_logits(cls_logits, labels)
        distil_correct += accuracy_from_logits(distil_logits, labels)
        total += labels.size(0)

    train_loss = running_loss / total
    train_cls_acc = 100 * cls_correct / total
    train_distil_acc = 100 * distil_correct / total

    # ---------------- VALIDATION ----------------
    student.eval()

    val_loss_total = 0.0
    val_cls_correct = 0
    val_distil_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_dl:
            images = images.to(device)
            labels = labels.to(device)

            teacher_logits = teacher(images)
            cls_logits, distil_logits = student(images)

            loss = kd_loss(
                cls_logits,
                distil_logits,
                teacher_logits,
                labels
            )

            val_loss_total += loss.item() * images.size(0)
            val_cls_correct += accuracy_from_logits(cls_logits, labels)
            val_distil_correct += accuracy_from_logits(distil_logits, labels)
            val_total += labels.size(0)

    val_loss = val_loss_total / val_total
    val_cls_acc = 100 * val_cls_correct / val_total
    val_distil_acc = 100 * val_distil_correct / val_total

    # DeiT inference commonly averages the two heads.
    # ADDED: this gives one final student accuracy using both tokens.
    val_combined_correct = 0

    with torch.no_grad():
        for images, labels in val_dl:
            images = images.to(device)
            labels = labels.to(device)

            cls_logits, distil_logits = student(images)

            combined_logits = (cls_logits + distil_logits) / 2
            val_combined_correct += accuracy_from_logits(
                combined_logits, labels
            )

    val_combined_acc = 100 * val_combined_correct / val_total

    print(
        f"Student Epoch [{epoch+1}/{EPOCHS_STUDENT}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train CLS Acc: {train_cls_acc:.2f}% | "
        f"Train Distil Acc: {train_distil_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val CLS Acc: {val_cls_acc:.2f}% | "
        f"Val Distil Acc: {val_distil_acc:.2f}% | "
        f"Val Combined Acc: {val_combined_acc:.2f}%"
    )


Student Epoch [1/5] Train Loss: 1.2028 | Train CLS Acc: 82.37% | Train Distil Acc: 84.35% | Val Loss: 0.5224 | Val CLS Acc: 91.78% | Val Distil Acc: 93.44% | Val Combined Acc: 93.56%
Student Epoch [2/5] Train Loss: 0.4040 | Train CLS Acc: 93.99% | Train Distil Acc: 94.77% | Val Loss: 0.3278 | Val CLS Acc: 95.12% | Val Distil Acc: 95.86% | Val Combined Acc: 95.84%
Student Epoch [3/5] Train Loss: 0.2963 | Train CLS Acc: 95.82% | Train Distil Acc: 96.27% | Val Loss: 0.2834 | Val CLS Acc: 96.20% | Val Distil Acc: 96.19% | Val Combined Acc: 96.50%
Student Epoch [4/5] Train Loss: 0.2420 | Train CLS Acc: 96.55% | Train Distil Acc: 96.91% | Val Loss: 0.2286 | Val CLS Acc: 96.98% | Val Distil Acc: 96.93% | Val Combined Acc: 97.08%
Student Epoch [5/5] Train Loss: 0.2129 | Train CLS Acc: 97.00% | Train Distil Acc: 97.32% | Val Loss: 0.2338 | Val CLS Acc: 97.11% | Val Distil Acc: 96.87% | Val Combined Acc: 97.17%


## Final Test Accuracy ADDED: Since `val_ds` is MNIST's official test split, this evaluates the final student on that split.

In [22]:
# ADDED: Final student test accuracy

student.eval()

correct_cls = 0
correct_distil = 0
correct_combined = 0
total = 0

with torch.no_grad():
    for images, labels in val_dl:
        images = images.to(device)
        labels = labels.to(device)

        cls_logits, distil_logits = student(images)

        combined_logits = (cls_logits + distil_logits) / 2

        correct_cls += accuracy_from_logits(cls_logits, labels)
        correct_distil += accuracy_from_logits(distil_logits, labels)
        correct_combined += accuracy_from_logits(combined_logits, labels)
        total += labels.size(0)

print(f"Final CLS token accuracy      : {100 * correct_cls / total:.2f}%")
print(f"Final Distillation accuracy   : {100 * correct_distil / total:.2f}%")
print(f"Final Combined accuracy       : {100 * correct_combined / total:.2f}%")


Final CLS token accuracy      : 97.11%
Final Distillation accuracy   : 96.87%
Final Combined accuracy       : 97.17%
